In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])
train_dataset = datasets.MNIST(
    root = './data',
    train = True,
    download = True,
    transform = transform
)
test_dataset = datasets.MNIST(
    root='./data',
    train=False,             
    download=True,
    transform=transform
)
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)
print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(784,256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128,10)
        )
    def forward(self,x):
        x = self.flatten(x)
        logits = self.network(x)
        return logits
model = NeuralNetwork().to(DEVICE)
print('\n模型结构')
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n总参数量:{total_params:,}")
print(f"可训练参数量: {trainable_params:,}")
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=LEARNING_RATE)
def train(epoch,model,device,train_loader,optimizer,loss_fn):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx,(data,target) in enumerate(train_loader):
        data,target = data.to(device),target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output,target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _,predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

        if batch_idx % 200 == 0:
            print(f"  Epoch [{epoch}/{EPOCHS}] "
                  f"Batch [{batch_idx}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}")
    avg_loss = running_loss / len(train_loader)
    accuracy = 100. * correct / total
    print(f"  => 训练集 - 平均Loss: {avg_loss:.4f}, 准确率: {accuracy:.2f}%")
def test(model,device,test_loader,loss_fn):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for data,target in test_loader:
            data,target = data.to(device),target.to(device)
            output = model(data)
            test_loss += loss_fn(output,target).item()

            _,predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    avg_loss = test_loss / len(test_loader)
    accuracy = 100. * correct / total
    print(f"  => 测试集 - 平均Loss: {avg_loss:.4f}, 准确率: {accuracy:.2f}%\n")
    return accuracy
print("\n========== 开始训练 ==========")
best_accuracy = 0.0
for epoch in range(1,EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
    train(epoch,model,DEVICE,train_loader,optimizer,loss_fn)
    accuracy = test(model,DEVICE,test_loader,loss_fn)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(),'best_mnist_model.pth')
        print(f"  ★ 保存最佳模型 (准确率: {best_accuracy:.2f}%)")
print(f"\n========== 训练完成! 最佳测试准确率: {best_accuracy:.2f}% ==========")
print("\n========== 预测演示 ==========")
model.load_state_dict(torch.load('best_mnist_model.pth',weights_only=True))
model.eval()
sample_data, sample_target = test_dataset[0]
sample_input = sample_data.unsqueeze(0).to(DEVICE)
with torch.no_grad():
    output = model(sample_input)
    probabilities = torch.softmax(output,dim=1)
    predicted_class = output.argmax(dim=1).item()
    confidence = probabilities[0][predicted_class].item()*100
print(f"真实标签: {sample_target}")
print(f"预测结果: {predicted_class}")
print(f"置信度:   {confidence:.2f}%")
print(f"\n各类别概率:")
for i, prob in enumerate(probabilities[0]):
    bar = "█" * int(prob.item() * 50)
    print(f"  数字 {i}: {prob.item():.4f} {bar}")  

cuda
训练集大小: 60000
测试集大小: 10000

模型结构
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=10, bias=True)
  )
)

总参数量:235,146
可训练参数量: 235,146

========== 开始训练 ==========

--- Epoch 1/10 ---
  Epoch [1/10] Batch [0/938] Loss: 2.3148
  Epoch [1/10] Batch [200/938] Loss: 0.2071
  Epoch [1/10] Batch [400/938] Loss: 0.1486
  Epoch [1/10] Batch [600/938] Loss: 0.1220
  Epoch [1/10] Batch [800/938] Loss: 0.1211
  => 训练集 - 平均Loss: 0.2735, 准确率: 91.65%
  => 测试集 - 平均Loss: 0.1141, 准确率: 96.42%

  ★ 保存最佳模型 (准确率: 96.42%)

--- Epoch 2/10 ---
  Epoch [2/10] Batch [0/938] Loss: 0.1651
  Epoch [2/10] Batch [200/938] Loss: 0.1636
  Epoch [2/10] Batch [400/938] Loss: 0.1994
  Epoch [2/10] Batch [600/

In [7]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report,confusion_matrix
import pandas as pd
iris = load_iris()
X = iris.data
y = iris.target
print(f'特征矩阵形状:{X.shape}')
print(f'标签分布:\n{pd.Series(y).value_counts().sort_index()}')
print(f'类别名称:{iris.target_names}')
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=43,
    stratify=y
)
print(f'\n训练集:{X_train.shape},测试集:{X_test.shape}')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = DecisionTreeClassifier(max_depth=3,random_state=42)
clf.fit(X_train_scaled,y_train)

y_pred = clf.predict(X_test_scaled)

print(f"\n=== 准确率 ===")
print(f"训练集:{clf.score(X_train_scaled,y_train):.4f}")
print(f"测试集:{clf.score(X_test_scaled,y_test):.4f}")

print(f"\n=== 分类报告 ===")
print(classification_report(y_test,y_pred,target_names=iris.target_names))
print(f"=== 混淆矩阵 ===")
print(confusion_matrix(y_test,y_pred))

特征矩阵形状:(150, 4)
标签分布:
0    50
1    50
2    50
Name: count, dtype: int64
类别名称:['setosa' 'versicolor' 'virginica']

训练集:(120, 4),测试集:(30, 4)

=== 准确率 ===
训练集:0.9750
测试集:0.9667

=== 分类报告 ===
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.91      1.00      0.95        10
   virginica       1.00      0.90      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

=== 混淆矩阵 ===
[[10  0  0]
 [ 0 10  0]
 [ 0  1  9]]


In [9]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
data = load_breast_cancer()
X_train,X_test,y_train,y_test = train_test_split(
    data.data,data.target,test_size=0.2,random_state=42,stratify=data.target
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
models = {
    "逻辑回归": LogisticRegression(max_iter=10000, random_state=42),
    "决策树": DecisionTreeClassifier(max_depth=5, random_state=42),
    "随机森林": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel="rbf", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}
results = []
for name,model in models.items():
    model.fit(X_train,y_train)
    train_acc = model.score(X_train,y_train)
    test_acc = model.score(X_test,y_test)
    results.append({
        "模型": name,
        "训练集准确率": f"{train_acc:.4f}",
        "测试集准确率": f"{test_acc:.4f}",
        "过拟合程度": f"{train_acc - test_acc:.4f}",
    })
    df_results = pd.DataFrame(results)
    print("=== 模型对比 ===")
    print(df_results.to_string(index=False))
    best = df_results.loc[df_results['测试集准确率'].astype(float).idxmax()]
    print(f"\n🏆 最佳模型: {best['模型']} (测试集准确率: {best['测试集准确率']})")

=== 模型对比 ===
  模型 训练集准确率 测试集准确率   过拟合程度
逻辑回归 0.9560 0.9649 -0.0089

🏆 最佳模型: 逻辑回归 (测试集准确率: 0.9649)
=== 模型对比 ===
  模型 训练集准确率 测试集准确率   过拟合程度
逻辑回归 0.9560 0.9649 -0.0089
 决策树 0.9934 0.9211  0.0724

🏆 最佳模型: 逻辑回归 (测试集准确率: 0.9649)
=== 模型对比 ===
  模型 训练集准确率 测试集准确率   过拟合程度
逻辑回归 0.9560 0.9649 -0.0089
 决策树 0.9934 0.9211  0.0724
随机森林 1.0000 0.9561  0.0439

🏆 最佳模型: 逻辑回归 (测试集准确率: 0.9649)
=== 模型对比 ===
  模型 训练集准确率 测试集准确率   过拟合程度
逻辑回归 0.9560 0.9649 -0.0089
 决策树 0.9934 0.9211  0.0724
随机森林 1.0000 0.9561  0.0439
 SVM 0.9187 0.9298 -0.0111

🏆 最佳模型: 逻辑回归 (测试集准确率: 0.9649)
=== 模型对比 ===
  模型 训练集准确率 测试集准确率   过拟合程度
逻辑回归 0.9560 0.9649 -0.0089
 决策树 0.9934 0.9211  0.0724
随机森林 1.0000 0.9561  0.0439
 SVM 0.9187 0.9298 -0.0111
 KNN 0.9473 0.9123  0.0350

🏆 最佳模型: 逻辑回归 (测试集准确率: 0.9649)


In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
import numpy as np
import pandas as pd
california = fetch_california_housing()
X = california.data
y = california.target
print(f"特征: {california.feature_names}")
print(f"样本数: {X.shape[0]}, 特征数: {X.shape[1]}")
print(f"房价范围: ${y.min():.1f} ~ ${y.max():.1f} (10万美元)")

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

regressors ={
    '线性回归':LinearRegression(),
    '岭回归':Ridge(alpha=1.0),
    '随机森林':RandomForestRegressor(n_estimators=100,random_state=42),
}
results = []
for name,model in regressors.items():
    model.fit(X_train_s,y_train)
    y_pred = model.predict(X_test_s)
    results.append({
        "模型": name,
        "MAE": f"{mean_absolute_error(y_test, y_pred):.4f}",
        "RMSE": f"{np.sqrt(mean_squared_error(y_test, y_pred)):.4f}",
        "R²": f"{r2_score(y_test, y_pred):.4f}",
    })
print("\n=== 回归模型对比 ===")
print(pd.DataFrame(results).to_string(index=False))
rf = regressors['随机森林']
feature_importance = pd.DataFrame({
    "特征": california.feature_names,
    "重要性": rf.feature_importances_,
}).sort_values('重要性',ascending=False)
print(f"\n=== 特征重要性（随机森林）===")
print(feature_importance.to_string(index=False))

特征: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
样本数: 20640, 特征数: 8
房价范围: $0.1 ~ $5.0 (10万美元)

=== 回归模型对比 ===
  模型    MAE   RMSE     R²
线性回归 0.5332 0.7456 0.5758
 岭回归 0.5332 0.7456 0.5758
随机森林 0.3278 0.5059 0.8047

=== 特征重要性（随机森林）===
        特征      重要性
    MedInc 0.524980
  AveOccup 0.138367
  Latitude 0.089035
 Longitude 0.088474
  HouseAge 0.054602
  AveRooms 0.044216
Population 0.030662
 AveBedrms 0.029663
